<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Model_Pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model_Pruning
Structured + Unstructured pruning sweep over MobileNetV2 **and** MobileNetV3.

**Experiments:** 2 architectures × 2 prune types × 3 sparsity levels = 12 runs.  
All runs share the same fine-tune budget (10 epochs). A results table is printed at the end.  
Pick the best row — that checkpoint feeds directly into `Pipeline_Prune.ipynb`.

> **Structured pruning + depthwise conv — important caveat:**  
> Depthwise layers (`groups == in_channels`) are **skipped** for structured pruning.  
> Dropping a whole filter from a depthwise layer breaks channel alignment with the  
> adjacent pointwise layer inside each MobileNet inverted-residual block.  
> Structured pruning therefore applies only to **pointwise** (`groups == 1`) Conv2d layers.  
> Unstructured pruning applies to **all** Conv2d layers as before.

> **STM32 / X-CUBE-AI note:**  
> Unstructured sparsity does **not** automatically shrink runtime or memory on STM32  
> unless your X-CUBE-AI version supports sparse-weight inference.  
> Structured pruning produces a smaller dense model that will always be faster.  
> Keep this in mind when choosing the winner.

Run `Model_MobileNetV2` and `Model_MobileNetV3` first so the best-seed checkpoints exist on Drive.

In [2]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

Mounted at /content/drive
✅ utils loaded from Drive


In [3]:
# ── Imports ─────────────────────────────────────────────────────────
import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import VWW_MobileNetV2, VWW_MobileNetV3
from utils.train   import setup_device, set_seed, evaluate, train_epoch, validate_epoch

device = setup_device(seed=41)

Device: cuda


In [4]:
# ── Config ──────────────────────────────────────────────────────────
SAVE_DIR = "/content/drive/My Drive/stm32-thesis/checkpoints"

# Adjust seed suffixes to whichever seed gave the best val acc in each training run
CKPTS = {
    "MobileNetV2": f"{SAVE_DIR}/mobilenetv2_seed_74.pth",
    "MobileNetV3": f"{SAVE_DIR}/mobilenetv3_seed_74.pth",
}

MODEL_FNS = {
    "MobileNetV2": VWW_MobileNetV2,
    "MobileNetV3": VWW_MobileNetV3,
}

PRUNE_AMOUNTS = [0.10, 0.20, 0.30]   # 10 / 20 / 30 %
PRUNE_TYPES   = ["unstructured", "structured"]

FT_EPOCHS = 10
FT_LR     = 1e-4

In [5]:
# ── Dataset ──────────────────────────────────────────────────────────
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")

1/4 Download
⬇️  Downloading VWW archive...
✅ Download complete: /content/vww_work/vw_coco2014_96.tar.gz
2/4 Extract
📦 Extracting VWW archive...
✅ Extraction complete: /content/vww_work/extracted
3/4 Find root
   Root: /content/vww_work/extracted/vw_coco2014_96
4/4 Manifests
🧩 Creating fixed train/val/test split manifests...
✅ Train: 7000 | Val: 1500 | Test: 1500
   Manifests saved at: /content/drive/My Drive/vww_fixed_split_manifests
Train: 7000 | Val: 1500 | Batch: 64


In [6]:
# ── Pruning helpers ─────────────────────────────────────────────────────

def apply_unstructured(model, amount):
    """L1 unstructured: zeros individual weights in ALL Conv2d layers."""
    params = [(m, "weight") for m in model.modules() if isinstance(m, nn.Conv2d)]
    for layer, p in params:
        prune.l1_unstructured(layer, name=p, amount=amount)
    return params


def apply_structured(model, amount):
    """
    L2 structured (filter pruning): removes whole output filters (dim=0).
    Skips depthwise Conv2d (groups == in_channels) to avoid breaking
    channel alignment inside inverted-residual blocks.
    """
    params = []
    for m in model.modules():
        if isinstance(m, nn.Conv2d) and m.groups == 1:  # pointwise only
            prune.ln_structured(m, name="weight", amount=amount, n=2, dim=0)
            params.append((m, "weight"))
    return params


def remove_pruning_masks(params):
    """Make pruning permanent before saving the checkpoint."""
    for layer, p in params:
        prune.remove(layer, p)


def compute_sparsity(model):
    """Fraction of zero weights across all Conv2d layers."""
    total = zeroed = 0
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            w = m.weight.detach().cpu().numpy()
            total  += w.size
            zeroed += (w == 0).sum()
    return zeroed / total if total > 0 else 0.0

In [7]:
# ── Fine-tune helper ───────────────────────────────────────────────────

def fine_tune(model, ft_epochs, ft_lr, train_loader, val_loader, device):
    set_seed(41)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=ft_lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ft_epochs)

    best_acc   = 0.0
    best_state = None

    for epoch in range(1, ft_epochs + 1):
        tl, ta = train_epoch(model, train_loader, optimizer, criterion, device)
        _,  va = validate_epoch(model, val_loader, criterion, device)
        scheduler.step()

        marker = " ✅" if va > best_acc else ""
        print(f"  FT {epoch:2d}/{ft_epochs} | train {ta*100:.2f}% | val {va*100:.2f}%{marker}")

        if va > best_acc:
            best_acc   = va
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return best_acc

In [8]:
# ── Main sweep ──────────────────────────────────────────────────────────
# 2 models x 2 prune types x 3 amounts = 12 experiments.
# Each: fresh model load -> prune -> evaluate -> fine-tune -> save checkpoint.

records = []

for model_name, model_fn in MODEL_FNS.items():
    ckpt_path = CKPTS[model_name]
    if not os.path.exists(ckpt_path):
        print(f"⚠️  Checkpoint not found, skipping {model_name}: {ckpt_path}")
        continue

    # Baseline accuracy (load once per architecture)
    base_model = model_fn().to(device)
    base_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    base_acc = evaluate(base_model, val_loader, device)
    print(f"\n{'='*62}")
    print(f"{model_name}  baseline val acc: {base_acc*100:.2f}%")
    print(f"{'='*62}")
    del base_model

    for prune_type in PRUNE_TYPES:
        for amount in PRUNE_AMOUNTS:
            tag = f"{model_name} | {prune_type:12s} | {int(amount*100):2d}%"
            print(f"\n▶ {tag}")

            # Fresh weights every run
            model = model_fn().to(device)
            model.load_state_dict(torch.load(ckpt_path, map_location=device))

            if prune_type == "unstructured":
                applied = apply_unstructured(model, amount)
            else:
                applied = apply_structured(model, amount)

            post_prune_acc   = evaluate(model, val_loader, device)
            actual_sparsity  = compute_sparsity(model)
            print(f"  After pruning (no FT): {post_prune_acc*100:.2f}%  "
                  f"actual sparsity: {actual_sparsity*100:.1f}%")

            t0 = time.time()
            ft_acc = fine_tune(model, FT_EPOCHS, FT_LR, train_loader, val_loader, device)
            elapsed = (time.time() - t0) / 60

            # Permanently bake in pruning masks, then save
            remove_pruning_masks(applied)
            save_name = (
                f"{model_name.lower()}_pruned_"
                f"{prune_type}_{int(amount*100)}pct.pth"
            )
            save_path = f"{SAVE_DIR}/{save_name}"
            torch.save(model.state_dict(), save_path)

            acc_delta = (ft_acc - base_acc) * 100
            print(f"  After FT: {ft_acc*100:.2f}%  "
                  f"delta vs baseline: {acc_delta:+.2f}%  "
                  f"({elapsed:.1f} min)")
            print(f"  Saved → {save_name}")

            records.append({
                "model":             model_name,
                "prune_type":        prune_type,
                "target_%":          int(amount * 100),
                "actual_sparsity_%": round(actual_sparsity * 100, 1),
                "baseline_%":        round(base_acc * 100, 2),
                "post_prune_%":      round(post_prune_acc * 100, 2),
                "after_FT_%":        round(ft_acc * 100, 2),
                "delta_%":           round(acc_delta, 2),
                "save_path":         save_path,
            })

            del model

print("\n✅ Sweep complete.")


MobileNetV2  baseline val acc: 78.40%

▶ MobileNetV2 | unstructured | 10%
  After pruning (no FT): 78.47%  actual sparsity: 10.0%
  FT  1/10 | train 81.39% | val 78.33% ✅
  FT  2/10 | train 82.00% | val 78.27%
  FT  3/10 | train 81.63% | val 78.27%
  FT  4/10 | train 81.51% | val 77.93%
  FT  5/10 | train 82.29% | val 78.40% ✅
  FT  6/10 | train 81.91% | val 78.73% ✅
  FT  7/10 | train 82.09% | val 78.93% ✅
  FT  8/10 | train 82.73% | val 78.60%
  FT  9/10 | train 82.26% | val 78.60%
  FT 10/10 | train 82.74% | val 78.67%
  After FT: 78.93%  delta vs baseline: +0.53%  (2.8 min)
  Saved → mobilenetv2_pruned_unstructured_10pct.pth

▶ MobileNetV2 | unstructured | 20%
  After pruning (no FT): 77.27%  actual sparsity: 20.0%
  FT  1/10 | train 81.26% | val 78.00% ✅
  FT  2/10 | train 81.80% | val 78.33% ✅
  FT  3/10 | train 81.20% | val 77.87%
  FT  4/10 | train 81.51% | val 77.93%
  FT  5/10 | train 82.07% | val 77.80%
  FT  6/10 | train 81.56% | val 77.60%
  FT  7/10 | train 82.14% | val 

In [9]:
# ── Results table ──────────────────────────────────────────────────────
df = pd.DataFrame(records)
df = df.sort_values(["model", "prune_type", "target_%"]).reset_index(drop=True)

display_cols = [
    "model", "prune_type", "target_%", "actual_sparsity_%",
    "baseline_%", "post_prune_%", "after_FT_%", "delta_%",
]
print("\n" + "="*82)
print("PRUNING SWEEP RESULTS")
print("="*82)
print(df[display_cols].to_string(index=False))
print("="*82)

# Best run by final val accuracy
best_row = df.loc[df["after_FT_%"].idxmax()]
print(f"\n🏆 Best: {best_row['model']} | {best_row['prune_type']} | "
      f"{int(best_row['target_%'])}%  →  {best_row['after_FT_%']}%  "
      f"(delta {best_row['delta_%']:+.2f}%)")
print(f"   Checkpoint: {best_row['save_path']}")
print("\n→ Copy that filename into Pipeline_Prune.ipynb (CKPT variable).")


PRUNING SWEEP RESULTS
      model   prune_type  target_%  actual_sparsity_%  baseline_%  post_prune_%  after_FT_%  delta_%
MobileNetV2   structured        10                8.9       78.40         61.20       78.60     0.20
MobileNetV2   structured        20               18.5       78.40         50.13       76.53    -1.87
MobileNetV2   structured        30               27.8       78.40         50.00       73.40    -5.00
MobileNetV2 unstructured        10               10.0       78.40         78.47       78.93     0.53
MobileNetV2 unstructured        20               20.0       78.40         77.27       78.53     0.13
MobileNetV2 unstructured        30               30.0       78.40         71.33       78.27    -0.13
MobileNetV3   structured        10                9.4       79.13         55.13       77.87    -1.27
MobileNetV3   structured        20               19.0       79.13         50.00       76.67    -2.47
MobileNetV3   structured        30               28.3       79.13   

## Next step
Copy the winning checkpoint filename printed above into `Pipeline_Prune.ipynb`:

```python
CKPT    = f"{SAVE_DIR}/<winning_checkpoint>.pth"
OUT_DIR = Path("/content/drive/My Drive/stm32-thesis/exports/pruned_<tag>")
```

Also update the model class in `Pipeline_Prune.ipynb` if the winner is MobileNetV3:

```python
# Change this line:
model = VWW_MobileNetV2().to(device)
# To:
model = VWW_MobileNetV3().to(device)
```

### Structured vs Unstructured on STM32
| | Unstructured | Structured |
|---|---|---|
| Accuracy recovery | Usually better | Harder (larger capacity loss) |
| Real speedup on STM32 | Only if X-CUBE-AI supports sparse ops | Yes — smaller dense model |
| Model file size | Unchanged (zeros still stored) | Smaller `.tflite`/ONNX |

If X-CUBE-AI does **not** exploit unstructured sparsity in your version,  
a structured model at 20–30% will be both faster and smaller on-device  
even if its val accuracy is slightly lower.